In [1]:
import json
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from matplotlib import animation
from IPython.display import HTML, Video, display
import tqdm as tqdm
from skimage.feature import local_binary_pattern
from scipy.spatial.distance import cdist
from scipy.spatial.distance import pdist
import os

In [ ]:
df = pd.read_csv("/home/guilherme_monteiro/projetos/tcc/data/bronze/manifests/video-metadata-publish-with-links.csv")
df['video_path'] = df['Filename'].apply(lambda x: "/home/guilherme_monteiro/projetos/tcc/data/bronze/videos/" + x)
df_true = df[df['Video Ground Truth'] == "Real"].reset_index(drop=True)
df_fake = df[df['Video Ground Truth'] == "Fake"].reset_index(drop=True)

def metadata_path_for_video(video_path):
    video_stem = os.path.splitext(os.path.basename(video_path))[0]
    return os.path.join(
        "/home/guilherme_monteiro/projetos/tcc/data/silver/face_metadata_json",
        f"{video_stem}_meta.json"
    )

df['metadata_path'] = df['video_path'].apply(metadata_path_for_video)
df['has_metadata'] = df['metadata_path'].apply(os.path.exists)
df_meta = df[df['has_metadata']].reset_index(drop=True)

print(f"Vídeos no CSV: {len(df)}")
print(f"Vídeos com metadata disponível: {len(df_meta)}")
print(df_meta['Video Ground Truth'].value_counts())


# Funções básicas

In [ ]:
def get_video_frame_count(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return frame_count


def sample_frame_indices(frame_count, max_frames=None):
    if frame_count <= 0:
        return np.array([], dtype=int)

    if max_frames is None or frame_count <= max_frames:
        return np.arange(frame_count, dtype=int)

    return np.linspace(0, frame_count - 1, int(max_frames)).astype(int)


def load_video_frames(video_path, max_frames=None, return_indices=False):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = sample_frame_indices(frame_count, max_frames)

    frames = []
    used_indices = []

    for frame_idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()

        if not ret:
            continue

        frames.append(frame)
        used_indices.append(int(frame_idx))

    cap.release()

    frames = np.array(frames)

    if return_indices:
        return frames, np.array(used_indices, dtype=int), frame_count

    return frames


def standardize_frame(frame, max_size=640):
    h, w = frame.shape[:2]

    scale = min(max_size / max(h, w), 1.0)
    new_w, new_h = int(w * scale), int(h * scale)

    frame_resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

    return frame_resized, scale


def load_metadata(video_path):
    full_path = metadata_path_for_video(video_path)

    with open(full_path, "r") as f:
        return json.load(f)


def metadata_index_for_frame(frame_idx, frame_count, metadata_len):
    if metadata_len <= 0:
        return None

    # Quando a metadata tem quase o mesmo tamanho do vídeo, ela é frame-a-frame.
    if metadata_len >= frame_count - 3:
        return min(int(frame_idx), metadata_len - 1)

    # Quando a metadata foi criada com amostragem, mapeia para o frame amostrado mais próximo.
    metadata_frame_indices = sample_frame_indices(frame_count, metadata_len)
    return int(np.argmin(np.abs(metadata_frame_indices - int(frame_idx))))


def scale_bbox(bbox, scale):
    return [int(round(x * scale)) for x in bbox]


def clip_bbox(bbox, width, height, min_size=4):
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]

    x1 = max(0, min(width - 1, x1))
    x2 = max(0, min(width, x2))
    y1 = max(0, min(height - 1, y1))
    y2 = max(0, min(height, y2))

    if x2 <= x1:
        x2 = min(width, x1 + min_size)
    if y2 <= y1:
        y2 = min(height, y1 + min_size)

    return [x1, y1, x2, y2]


def create_face_regions(frame, bbox, padding=0.2):
    h, w = frame.shape[:2]

    x1, y1, x2, y2 = clip_bbox(bbox, w, h)

    bw = x2 - x1
    bh = y2 - y1

    px = int(bw * padding)
    py = int(bh * padding)

    x1p = max(0, x1 - px)
    y1p = max(0, y1 - py)
    x2p = min(w, x2 + px)
    y2p = min(h, y2 + py)

    face_mask = np.zeros((h, w), dtype=np.uint8)
    face_mask[y1:y2, x1:x2] = 1

    expanded_mask = np.zeros((h, w), dtype=np.uint8)
    expanded_mask[y1p:y2p, x1p:x2p] = 1

    border_mask = expanded_mask - face_mask
    background_mask = 1 - expanded_mask

    return {
        "face": face_mask,
        "border": border_mask,
        "background": background_mask,
        "bbox": (x1, y1, x2, y2),
        "bbox_expanded": (x1p, y1p, x2p, y2p)
    }


## Funções pra visualização

In [4]:
def draw_text_block(img, text_lines, x=10, y=20, line_height=18):
    for i, line in enumerate(text_lines):
        cv2.putText(
            img,
            line,
            (x, y + i * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
    return img

# Desenha a caixa delimitadora no frame
def draw_face_box(frame, bbox, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = bbox
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def overlay_regions(frame, regions):
    overlay = frame.copy()

    face_mask = regions["face"]
    border_mask = regions["border"]
    background_mask = regions["background"]

    overlay[face_mask == 1] = (0, 255, 0)       # verde
    overlay[border_mask == 1] = (0, 0, 255)     # vermelho
    overlay[background_mask == 1] = (255, 0, 0) # azul

    alpha = 0.3
    return cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

# FFT

A versão robusta calcula FFT em crops espaciais reais: face, borda local e fundo local. A versão anterior aplicava máscaras espaciais diretamente sobre o espectro do frame inteiro, o que misturava posição da imagem com coordenadas de frequência.


In [ ]:
def compute_fft(gray, fft_size=128, remove_mean=True, apply_window=True):
    if gray is None or gray.size == 0:
        return None

    if gray.ndim == 3:
        gray = cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)

    gray = cv2.resize(gray, (fft_size, fft_size), interpolation=cv2.INTER_AREA)
    gray = gray.astype(np.float32)

    if remove_mean:
        gray = gray - np.mean(gray)

    if apply_window:
        window = np.outer(np.hanning(gray.shape[0]), np.hanning(gray.shape[1])).astype(np.float32)
        gray = gray * window

    spectrum = np.fft.fftshift(np.fft.fft2(gray))
    power = np.abs(spectrum) ** 2

    return power


In [ ]:
def create_fft_masks(shape):
    h, w = shape
    cy, cx = h // 2, w // 2

    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2)
    max_r = dist.max()

    dc_mask = dist <= 1
    low_mask = (dist > 1) & (dist <= 0.25 * max_r)
    mid_mask = (dist > 0.25 * max_r) & (dist <= 0.60 * max_r)
    high_mask = dist > 0.60 * max_r

    bins = np.linspace(-np.pi, np.pi, 9)
    angle = np.arctan2(Y - cy, X - cx)
    angle_masks = [((angle >= bins[i]) & (angle < bins[i + 1])) for i in range(len(bins) - 1)]

    return {
        "dc": dc_mask,
        "low": low_mask,
        "mid": mid_mask,
        "high": high_mask,
        "angle": angle_masks,
        "dist": dist,
        "max_r": max_r,
    }


In [ ]:
def compute_fft_region(region, fft_size=128):
    if region is None or region.size == 0:
        return None

    h, w = region.shape[:2]
    if h < 8 or w < 8:
        return None

    gray = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY) if region.ndim == 3 else region
    mean_intensity = float(np.mean(gray))
    std_intensity = float(np.std(gray))

    power = compute_fft(gray, fft_size=fft_size, remove_mean=True, apply_window=True)
    if power is None:
        return None

    masks = create_fft_masks(power.shape)
    total_energy = float(np.sum(power) + 1e-12)

    low_energy = float(np.sum(power[masks["low"]]) / total_energy)
    mid_energy = float(np.sum(power[masks["mid"]]) / total_energy)
    high_energy = float(np.sum(power[masks["high"]]) / total_energy)

    prob = power.ravel() / total_energy
    prob = prob[prob > 1e-12]
    entropy = float(-np.sum(prob * np.log(prob)) / np.log(power.size))

    angle_energy = np.array([np.sum(power[m]) for m in masks["angle"]], dtype=np.float64)
    anisotropy = float(np.std(angle_energy) / (np.mean(angle_energy) + 1e-12))

    radial_centroid = float(np.sum(power * masks["dist"]) / (total_energy * masks["max_r"] + 1e-12))
    geometric_mean = float(np.exp(np.mean(np.log(power + 1e-12))))
    arithmetic_mean = float(np.mean(power) + 1e-12)
    flatness = geometric_mean / arithmetic_mean

    return {
        "low_freq_ratio": low_energy,
        "mid_freq_ratio": mid_energy,
        "high_freq_ratio": high_energy,
        "entropy_norm": entropy,
        "anisotropy": anisotropy,
        "radial_centroid": radial_centroid,
        "flatness": flatness,
        "mean_intensity": mean_intensity,
        "std_intensity": std_intensity,
    }


def fft_region_differences(f1, f2, prefix):
    if f1 is None or f2 is None:
        return {}

    comparable_metrics = [
        "low_freq_ratio",
        "mid_freq_ratio",
        "high_freq_ratio",
        "entropy_norm",
        "anisotropy",
        "radial_centroid",
        "flatness",
        "std_intensity",
    ]

    return {
        f"{prefix}_fft_{metric}_diff": abs(f1[metric] - f2[metric])
        for metric in comparable_metrics
    }


In [ ]:
def _largest_valid_patch(patches):
    valid = [p for p in patches if p is not None and p.size > 0 and p.shape[0] >= 8 and p.shape[1] >= 8]
    if not valid:
        return None

    return max(valid, key=lambda p: p.shape[0] * p.shape[1])


def extract_fft_region_crops(frame, bbox, border_padding=0.25, background_padding=0.90):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = clip_bbox(bbox, w, h)

    face_crop = frame[y1:y2, x1:x2]

    bw = x2 - x1
    bh = y2 - y1

    bx1 = max(0, x1 - int(bw * border_padding))
    by1 = max(0, y1 - int(bh * border_padding))
    bx2 = min(w, x2 + int(bw * border_padding))
    by2 = min(h, y2 + int(bh * border_padding))

    border_patches = [
        frame[by1:y1, bx1:bx2],
        frame[y2:by2, bx1:bx2],
        frame[y1:y2, bx1:x1],
        frame[y1:y2, x2:bx2],
    ]
    border_crop = _largest_valid_patch(border_patches)

    cx1 = max(0, x1 - int(bw * background_padding))
    cy1 = max(0, y1 - int(bh * background_padding))
    cx2 = min(w, x2 + int(bw * background_padding))
    cy2 = min(h, y2 + int(bh * background_padding))

    background_patches = [
        frame[cy1:by1, cx1:cx2],
        frame[by2:cy2, cx1:cx2],
        frame[by1:by2, cx1:bx1],
        frame[by1:by2, bx2:cx2],
    ]
    background_crop = _largest_valid_patch(background_patches)

    return {
        "face": face_crop,
        "border": border_crop,
        "background": background_crop,
    }


def compute_fft_metrics(frame, bbox):
    frame_std, scale = standardize_frame(frame)
    h, w = frame_std.shape[:2]

    bbox_scaled = scale_bbox(bbox, scale)
    bbox_scaled = clip_bbox(bbox_scaled, w, h)

    crops = extract_fft_region_crops(frame_std, bbox_scaled)

    face = compute_fft_region(crops["face"])
    border = compute_fft_region(crops["border"])
    bg = compute_fft_region(crops["background"])

    features = {}

    for region_name, region_features in [("face", face), ("border", border), ("background", bg)]:
        if region_features is None:
            continue

        for metric_name, metric_value in region_features.items():
            features[f"fft_{region_name}_{metric_name}"] = metric_value

    features.update(fft_region_differences(face, bg, "face_bg"))
    features.update(fft_region_differences(face, border, "face_border"))
    features.update(fft_region_differences(border, bg, "border_bg"))

    face_power = compute_fft(crops["face"]) if crops["face"] is not None else None

    return features, face_power


In [ ]:
def fft_visual(mag):
    if mag is None:
        return np.zeros((128, 128), dtype=np.uint8)

    mag_log = np.log(mag + 1)
    mag_vis = cv2.normalize(mag_log, None, 0, 255, cv2.NORM_MINMAX)

    return mag_vis.astype(np.uint8)


def debug_fft_video(video_path, max_frames=120, interval=60):
    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)

    debug_frames = []

    print("\nProcessando FFT robusta...")

    for i, frame in enumerate(tqdm.tqdm(frames)):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        frame_std, scale = standardize_frame(frame)
        bbox = metadata[metadata_idx]["bbox"]
        bbox_scaled = scale_bbox(bbox, scale)
        bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])

        regions = create_face_regions(frame_std, bbox_scaled)
        features, mag = compute_fft_metrics(frame, bbox)
        fft_vis = fft_visual(mag)

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {frame_idx} | Meta: {metadata_idx}",
            f"Face High: {features.get('fft_face_high_freq_ratio', 0):.3f}",
            f"Face Entropy: {features.get('fft_face_entropy_norm', 0):.3f}",
            f"Face Aniso: {features.get('fft_face_anisotropy', 0):.3f}",
            f"Face-BG High Diff: {features.get('face_bg_fft_high_freq_ratio_diff', 0):.3f}",
            f"Face-BG Entropy Diff: {features.get('face_bg_fft_entropy_norm_diff', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(cv2.resize(fft_vis, (frame_std.shape[1], frame_std.shape[0])), cv2.COLOR_GRAY2RGB)
        ])

        debug_frames.append(combined)

    if not debug_frames:
        raise ValueError("Nenhum frame válido para visualização")

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")

    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(debug_frames),
        interval=interval,
        blit=True,
    )

    plt.close(fig)
    display(HTML(ani.to_jshtml()))


In [ ]:
# debug_fft_video(df_fake['video_path'][1], max_frames=200)

In [ ]:
# debug_fft_video(df_true['video_path'][0], max_frames=200)

# Todas as metricas

In [ ]:
def all_metrics(video_path, max_frames=500, label=None):
    video_name = os.path.basename(video_path)
    print(f"\nExtraindo métricas do vídeo: {video_name}")

    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)

    all_features = []

    for i, frame in enumerate(frames):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        bbox = metadata[metadata_idx]["bbox"]
        features_fft, _ = compute_fft_metrics(frame, bbox)

        features = {**features_fft}
        features["frame"] = frame_idx
        features["metadata_idx"] = metadata_idx

        all_features.append(features)

    metrics_df = pd.DataFrame(all_features)
    metrics_df.insert(0, "video_name", video_name)

    if label is not None:
        metrics_df.insert(1, "label", label)

    return metrics_df


def metadata_exists(video_path):
    return os.path.exists(metadata_path_for_video(video_path))


def aggregate_video_metrics(frame_metrics):
    metric_cols = [
        col for col in frame_metrics.columns
        if (col.startswith("fft_") or "_fft_" in col)
    ]

    agg = frame_metrics[metric_cols].agg(["mean", "std", "median"])
    values = {}

    for metric in metric_cols:
        values[f"{metric}_mean"] = agg.loc["mean", metric]
        values[f"{metric}_std"] = agg.loc["std", metric]
        values[f"{metric}_median"] = agg.loc["median", metric]

    return values


def auc_from_scores(labels, scores, positive_label="Fake"):
    valid = [(label, score) for label, score in zip(labels, scores) if pd.notna(score)]
    pos = [score for label, score in valid if label == positive_label]
    neg = [score for label, score in valid if label != positive_label]

    if len(pos) == 0 or len(neg) == 0:
        return np.nan

    wins = 0
    ties = 0

    for pos_score in pos:
        for neg_score in neg:
            wins += pos_score > neg_score
            ties += pos_score == neg_score

    return (wins + 0.5 * ties) / (len(pos) * len(neg))


def auc_abs_bootstrap_ci(labels, scores, positive_label="Fake", n_bootstrap=1000, ci=0.95, random_state=42):
    values = pd.DataFrame({"label": labels, "score": scores}).dropna()
    pos = values.loc[values["label"] == positive_label, "score"].astype(float).to_numpy()
    neg = values.loc[values["label"] != positive_label, "score"].astype(float).to_numpy()

    if len(pos) < 2 or len(neg) < 2:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(random_state)
    boot = []

    for _ in range(n_bootstrap):
        pos_sample = rng.choice(pos, size=len(pos), replace=True)
        neg_sample = rng.choice(neg, size=len(neg), replace=True)
        sample_labels = [positive_label] * len(pos_sample) + ["__negative__"] * len(neg_sample)
        sample_scores = np.concatenate([pos_sample, neg_sample])
        auc = auc_from_scores(sample_labels, sample_scores, positive_label=positive_label)
        if pd.notna(auc):
            boot.append(max(auc, 1 - auc))

    if not boot:
        return np.nan, np.nan, np.nan

    alpha = (1 - ci) / 2
    return (
        float(np.quantile(boot, alpha)),
        float(np.quantile(boot, 1 - alpha)),
        float(np.std(boot, ddof=1)) if len(boot) > 1 else 0.0,
    )


def cohen_d_by_label(values, labels, positive_label="Fake"):
    values = pd.Series(values)
    labels = pd.Series(labels)

    pos = values[labels == positive_label].dropna().astype(float)
    neg = values[labels != positive_label].dropna().astype(float)

    if len(pos) < 2 or len(neg) < 2:
        return np.nan

    pooled = np.sqrt((pos.var(ddof=1) + neg.var(ddof=1)) / 2)

    if pooled == 0 or pd.isna(pooled):
        return np.nan

    return (pos.mean() - neg.mean()) / pooled


def classify_metric_scope(metric):
    if "face_bg_" in metric:
        return "face_background_diff"
    if "border_bg_" in metric:
        return "border_background_diff"
    if "face_border_" in metric:
        return "face_border_diff"
    if "_background_" in metric:
        return "background_direct"
    if "_border_" in metric:
        return "border_direct"
    if "_face_" in metric:
        return "face_direct"
    return "other"


def is_context_sensitive_metric(metric):
    scope = classify_metric_scope(metric)
    return scope in {"background_direct", "face_background_diff", "border_background_diff"} or metric.endswith("patch_count_mean")


def build_metric_report(video_metrics, metric_cols, n_bootstrap=1000, random_state=42):
    report_rows = []

    for metric in metric_cols:
        valid = video_metrics[["label", metric]].dropna()
        auc = auc_from_scores(video_metrics["label"], video_metrics[metric])
        auc_abs = max(auc, 1 - auc) if pd.notna(auc) else np.nan
        ci_low, ci_high, auc_bootstrap_std = auc_abs_bootstrap_ci(
            video_metrics["label"],
            video_metrics[metric],
            n_bootstrap=n_bootstrap,
            random_state=random_state,
        )
        d = cohen_d_by_label(video_metrics[metric], video_metrics["label"])

        report_rows.append({
            "metric": metric,
            "scope": classify_metric_scope(metric),
            "context_sensitive": is_context_sensitive_metric(metric),
            "direction": "higher_fake" if pd.notna(auc) and auc >= 0.5 else "higher_real" if pd.notna(auc) else "undefined",
            "auc_fake": auc,
            "auc_abs": auc_abs,
            "auc_abs_ci_low": ci_low,
            "auc_abs_ci_high": ci_high,
            "auc_abs_bootstrap_std": auc_bootstrap_std,
            "cohen_d_fake_minus_real": d,
            "n_fake": int((valid["label"] == "Fake").sum()),
            "n_real": int((valid["label"] == "Real").sum()),
            "missing_rate": float(video_metrics[metric].isna().mean()),
            "real_mean": video_metrics.loc[video_metrics["label"] == "Real", metric].mean(),
            "fake_mean": video_metrics.loc[video_metrics["label"] == "Fake", metric].mean(),
        })

    report = pd.DataFrame(report_rows)

    if not report.empty:
        report = report.sort_values(
            ["auc_abs", "cohen_d_fake_minus_real"],
            ascending=[False, False],
            key=lambda s: s.abs() if s.name == "cohen_d_fake_minus_real" else s,
        ).reset_index(drop=True)

    return report


def evaluate_group_d(max_frames=120):
    rows = []
    available_df = df[df['video_path'].apply(metadata_exists)].reset_index(drop=True)

    for _, row in available_df.iterrows():
        frame_metrics = all_metrics(
            row['video_path'],
            max_frames=max_frames,
            label=row['Video Ground Truth']
        )

        if frame_metrics.empty:
            continue

        video_row = aggregate_video_metrics(frame_metrics)
        video_row["video_name"] = os.path.basename(row['video_path'])
        video_row["label"] = row['Video Ground Truth']
        video_row["n_frames"] = len(frame_metrics)
        rows.append(video_row)

    video_metrics = pd.DataFrame(rows)

    if video_metrics.empty:
        return video_metrics, pd.DataFrame()

    metric_cols = [
        col for col in video_metrics.columns
        if (col.startswith("fft_") or "_fft_" in col) and col.endswith("_mean")
    ]

    report = build_metric_report(video_metrics, metric_cols)

    print("\nVideos avaliados:")
    print(video_metrics['label'].value_counts())
    print("\nObservacao metodologica: auc_abs ranqueia separabilidade univariada; use direction e IC bootstrap antes de interpretar como sinal robusto.")

    return video_metrics, report


## Avaliação discriminativa

Executa a extração apenas nos vídeos com metadata disponível e resume uma linha por vídeo. Use `report.head(20)` para ver quais sinais separam melhor Real/Fake nesta amostra.

In [ ]:
# Rodada rápida de validação. Aumente max_frames para estabilizar os resultados.
video_metrics, report = evaluate_group_d(max_frames=120)
display(report.head(20))